# 01 - Dataset Preprocessing

Download Lichess standard rated games (1 month), parse PGN data, and produce:
1. **Structured CSV** with game metadata (ELO, time control, moves, openings, etc.)
2. **Blunder analysis CSV** with eval changes and clock times from games that have eval+clock annotations

Outputs are saved to Google Drive for reuse.

## Configuration
Set the month to download and processing parameters below.

In [ ]:
# ============================================================
# CONFIGURATION - Change these values as needed
# ============================================================

# Years and months to scan for games
# YEAR: list of years, e.g. [2023, 2024]
# MONTH: list of months (1-12), or None for all 12 months per year
YEAR = [2024]
MONTH = None  # e.g. [1, 6, 12] for specific months

# Total number of games to collect (evenly split across ELO brackets)
TOTAL_GAMES = 200_000

# ELO brackets for fair distribution
ELO_BRACKETS = [
    (0, 1000),
    (1000, 1400),
    (1400, 1800),
    (1800, 2200),
    (2200, 2600),
    (2600, 9999),
]

# Blunder threshold in centipawns (200 = same as Lichess)
BLUNDER_CUTOFF_CP = 200

# Sync parsed results to Drive every N games
SYNC_EVERY = 50_000

# Output file prefix
OUTPUT_PREFIX = "lichess_sampled"

## Setup
Install dependencies and mount Google Drive.

In [ ]:
import subprocess
import sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install("python-chess")
install("zstandard")
install("pandas")
install("tqdm")

print("Dependencies installed.")

In [ ]:
from google.colab import drive
import os
import shutil

drive.mount("/content/drive")

# Paths
DRIVE_ROOT = "/content/drive/MyDrive/Learning-document/Data Visualization/Project 2"
DRIVE_DATA_DIR = os.path.join(DRIVE_ROOT, "data")
VM_DATA_DIR = "/content/data"

os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
os.makedirs(VM_DATA_DIR, exist_ok=True)

print(f"Drive root:    {DRIVE_ROOT}")
print(f"Drive data:    {DRIVE_DATA_DIR}")
print(f"VM data dir:   {VM_DATA_DIR}")

## Download
Download the zstd-compressed PGN file from Lichess.

In [ ]:
def download_month(year, month):
    """Download zstd-compressed PGN file for a given month. Returns local path."""
    month_str = f"{year}-{month:02d}"
    filename = f"lichess_db_standard_rated_{month_str}.pgn.zst"
    url = f"https://database.lichess.org/standard/{filename}"
    local_path = f"/content/{filename}"

    if os.path.exists(local_path):
        print(f"File already exists: {local_path}")
        return local_path

    print(f"Downloading {url}...")
    print("This may take 10-30 minutes depending on file size.")
    urllib.request.urlretrieve(url, local_path)
    size_gb = os.path.getsize(local_path) / (1024**3)
    print(f"Downloaded: {size_gb:.1f} GB")
    return local_path

print("download_month() ready.")

## Streaming PGN Parser
Read the compressed PGN file line-by-line (no need to decompress to disk).
Extract both game metadata and move-level eval/clock data.

In [ ]:
import zstandard as zstd
import io
import chess
import chess.pgn
import re
from collections import defaultdict
from tqdm.notebook import tqdm
import pandas as pd
import time
import json
import urllib.request

print("Libraries loaded.")

In [ ]:
def open_zst_stream(filepath):
    """Open a .zst file for streaming line-by-line reading."""
    dctx = zstd.ZstdDecompressor()
    with open(filepath, 'rb') as fh:
        stream = dctx.stream_reader(fh)
        text_stream = io.TextIOWrapper(stream, encoding='utf-8')
        yield from text_stream


def read_games_from_stream(lines):
    """Yield individual game strings from a stream of PGN lines."""
    current_game = []
    for line in lines:
        line = line.rstrip('\n')
        if line == '' and current_game:
            has_movetext = any(not l.startswith('[') for l in current_game)
            if has_movetext:
                yield '\n'.join(current_game)
                current_game = []
            else:
                current_game.append(line)
        else:
            if line:
                current_game.append(line)
    if current_game:
        yield '\n'.join(current_game)


def parse_header(headers, key):
    """Safely extract a PGN header value."""
    try:
        return headers.get(key, None)
    except:
        return None


def time_to_seconds(clock_str):
    """Convert clock string like '0:01:30' to seconds."""
    if not clock_str:
        return None
    parts = clock_str.split(':')
    parts.reverse()
    total = 0
    mult = 1
    for p in parts:
        total += mult * float(p)
        mult *= 60
    return total


def parse_time_control(tc_str):
    """Parse time control like '300+0' or '600+5' into (base_seconds, increment)."""
    if not tc_str or '+' not in tc_str:
        return None, None
    try:
        parts = tc_str.split('+')
        return int(parts[0]), int(parts[1])
    except:
        return None, None


def extract_elos_from_text(game_text):
    """Quickly extract WhiteElo and BlackElo from raw PGN text (no python-chess)."""
    white_elo = None
    black_elo = None
    for line in game_text.split('\n'):
        if line.startswith('[WhiteElo "'):
            try:
                white_elo = int(line.split('"')[1])
            except (ValueError, IndexError):
                pass
        elif line.startswith('[BlackElo "'):
            try:
                black_elo = int(line.split('"')[1])
            except (ValueError, IndexError):
                pass
        if white_elo is not None and black_elo is not None:
            break
    return white_elo, black_elo


print("Parser functions defined.")

## Collect & Process Games
Phase 1: Fast-scan PGN files for games with eval annotations, bucket by ELO, and sample evenly.
Phase 2: Full parse of sampled games to extract metadata, moves, and blunders.

In [ ]:
def save_and_sync(games_metadata, blunder_records, move_records,
                  vm_dir, drive_dir, prefix):
    """Save DataFrames to VM disk, then copy to Google Drive."""
    df_g = pd.DataFrame(games_metadata)
    df_b = pd.DataFrame(blunder_records)
    df_m = pd.DataFrame(move_records)

    for suffix, df in [("games", df_g), ("blunders", df_b), ("moves", df_m)]:
        path = os.path.join(vm_dir, f"{prefix}_{suffix}.csv")
        df.to_csv(path, index=False)

    for suffix in ["games", "blunders", "moves"]:
        src = os.path.join(vm_dir, f"{prefix}_{suffix}.csv")
        dst = os.path.join(drive_dir, f"{prefix}_{suffix}.csv")
        if os.path.exists(src):
            shutil.copy2(src, dst)


def collect_sampled_games(filepath, buckets, elo_brackets, target_per_bracket):
    """
    Phase 1: Fast scan through a PGN file.
    Checks raw text for '%eval' (skips 95-98% of games instantly),
    extracts ELO from headers, and buckets qualifying games.
    Stops early when all brackets are full.
    """
    game_count = 0
    eval_count = 0
    start_time = time.time()

    for game_text in read_games_from_stream(open_zst_stream(filepath)):
        game_count += 1

        # Fast filter: skip games without eval (string check, no parsing)
        if '%eval' not in game_text:
            continue

        eval_count += 1

        # Quick ELO extraction (no python-chess)
        white_elo, black_elo = extract_elos_from_text(game_text)
        if white_elo is None or black_elo is None:
            continue

        avg_elo = (white_elo + black_elo) / 2

        # Bucket by ELO
        for bracket in elo_brackets:
            lo, hi = bracket
            if lo <= avg_elo < hi:
                if len(buckets[bracket]) < target_per_bracket:
                    buckets[bracket].append(game_text)
                break

        # Progress
        if game_count % 500_000 == 0:
            total_collected = sum(len(v) for v in buckets.values())
            elapsed = time.time() - start_time
            rate = game_count / elapsed
            print(f"  Scanned {game_count:,} ({rate:.0f}/s) | "
                  f"{eval_count:,} with eval | "
                  f"collected {total_collected:,}")

        # Early stop
        if all(len(buckets[b]) >= target_per_bracket for b in elo_brackets):
            break

    total_collected = sum(len(v) for v in buckets.values())
    elapsed = time.time() - start_time
    print(f"  Scan: {game_count:,} games in {elapsed:.1f}s, "
          f"{eval_count:,} with eval, collected {total_collected:,}")


def process_collected_games(game_texts, sync_every=None,
                            vm_dir=None, drive_dir=None, prefix=None):
    """
    Phase 2: Full python-chess parse of collected game texts.
    Extracts game metadata, move-level eval/clock data, and blunders.
    """
    games_metadata = []
    blunder_records = []
    move_records = []

    game_count = 0
    start_time = time.time()
    total = len(game_texts)

    for game_text in game_texts:
        game = chess.pgn.read_game(io.StringIO(game_text))
        if game is None:
            continue

        game_count += 1
        headers = game.headers

        event = parse_header(headers, 'Event')
        site = parse_header(headers, 'Site')
        white_elo = parse_header(headers, 'WhiteElo')
        black_elo = parse_header(headers, 'BlackElo')
        result = parse_header(headers, 'Result')
        time_control = parse_header(headers, 'TimeControl')
        termination = parse_header(headers, 'Termination')
        opening_eco = parse_header(headers, 'ECO')
        opening_name = parse_header(headers, 'Opening')

        if result == '1-0':
            winner = 'white'
        elif result == '0-1':
            winner = 'black'
        elif result == '1/2-1/2':
            winner = 'draw'
        else:
            winner = None

        base_secs, increment = parse_time_control(time_control)

        victory_status = None
        if termination:
            if 'Normal' in termination:
                victory_status = 'mate' if 'checkmate' in termination.lower() else 'resign'
            elif 'Time' in termination:
                victory_status = 'timeout'
            elif 'Abandoned' in termination:
                victory_status = 'abandoned'
            elif 'Rules infraction' in termination:
                victory_status = 'rules_infraction'

        moves_list = []
        board = game.board()
        node = game

        has_eval = False
        has_clock = False
        prev_eval = 0.0
        move_number = 0
        total_moves = 0

        while node.variations:
            next_node = node.variation(0)
            move = next_node.move
            san = board.san(move)
            moves_list.append(san)
            total_moves += 1

            eval_cp = None
            clock_secs = None

            eval_annotation = next_node.eval()
            if eval_annotation is not None:
                has_eval = True
                score = eval_annotation.white()
                if score.score() is not None:
                    eval_cp = score.score()
                elif score.mate() is not None:
                    eval_cp = 10000 if score.mate() > 0 else -10000

            clock_annotation = next_node.clock()
            if clock_annotation is not None:
                has_clock = True
                clock_secs = clock_annotation

            is_white = board.turn == chess.WHITE
            move_number += 1

            if eval_cp is not None and clock_secs is not None:
                if is_white:
                    eval_change = eval_cp - prev_eval
                else:
                    eval_change = prev_eval - eval_cp

                move_records.append({
                    'game_idx': game_count - 1,
                    'move_number': move_number,
                    'is_white': is_white,
                    'move_san': san,
                    'eval_cp': eval_cp,
                    'eval_change_for_player': eval_change,
                    'clock_secs_remaining': clock_secs,
                    'base_time_secs': base_secs,
                    'increment': increment,
                })

                if eval_change <= -BLUNDER_CUTOFF_CP:
                    was_winning = (is_white and prev_eval > 200) or (not is_white and prev_eval < -200)
                    still_winning = (is_white and eval_cp > 200) or (not is_white and eval_cp < -200)

                    if not (was_winning and still_winning):
                        blunder_records.append({
                            'game_idx': game_count - 1,
                            'move_number': move_number,
                            'is_white': is_white,
                            'move_san': san,
                            'eval_change_cp': eval_change,
                            'clock_secs_remaining': clock_secs,
                            'base_time_secs': base_secs,
                            'white_elo': int(white_elo) if white_elo else None,
                            'black_elo': int(black_elo) if black_elo else None,
                            'player_elo': int(white_elo) if is_white else int(black_elo) if black_elo else None,
                        })

                prev_eval = eval_cp

            board.push(move)
            node = next_node

        event_str = event or ''
        is_rated = 'Rated' in event_str

        game_type = None
        if base_secs is not None:
            total_est = base_secs + increment * 80 if increment else base_secs
            if total_est < 180:
                game_type = 'bullet'
            elif total_est < 480:
                game_type = 'blitz'
            elif total_est < 1500:
                game_type = 'rapid'
            else:
                game_type = 'classical'

        games_metadata.append({
            'game_id': site.split('/')[-1] if site else None,
            'rated': is_rated,
            'turns': total_moves,
            'victory_status': victory_status,
            'winner': winner,
            'white_id': parse_header(headers, 'White'),
            'white_rating': int(white_elo) if white_elo and white_elo != '?' else None,
            'black_id': parse_header(headers, 'Black'),
            'black_rating': int(black_elo) if black_elo and black_elo != '?' else None,
            'moves': ' '.join(moves_list),
            'increment_code': time_control,
            'base_time_secs': base_secs,
            'increment': increment,
            'game_type': game_type,
            'opening_eco': opening_eco,
            'opening_name': opening_name,
            'opening_ply': sum(1 for _ in game.mainline_moves()) // 2,
            'has_eval': has_eval,
            'has_clock': has_clock,
        })

        # Periodic sync
        if sync_every and game_count % sync_every == 0:
            elapsed = time.time() - start_time
            rate = game_count / elapsed
            save_and_sync(games_metadata, blunder_records, move_records,
                          vm_dir, drive_dir, prefix)
            print(f"  Parsed {game_count:,}/{total:,} ({rate:.0f}/s) - synced")

    # Final save
    if vm_dir:
        save_and_sync(games_metadata, blunder_records, move_records,
                      vm_dir, drive_dir, prefix)

    elapsed = time.time() - start_time
    print(f"\nDone: {game_count:,}/{total:,} games parsed in {elapsed:.1f}s")

    return games_metadata, blunder_records, move_records

In [ ]:
import itertools

years = YEAR if isinstance(YEAR, list) else [YEAR]
months = MONTH if MONTH is not None else list(range(1, 13))
if not isinstance(months, list):
    months = [months]

elo_brackets = ELO_BRACKETS
target_per_bracket = TOTAL_GAMES // len(elo_brackets)
buckets = {bracket: [] for bracket in elo_brackets}

# Phase 1: Scan months and collect games with eval, bucketed by ELO
for y, m in itertools.product(years, months):
    all_full = all(len(buckets[b]) >= target_per_bracket for b in elo_brackets)
    if all_full:
        break

    month_str = f"{y}-{m:02d}"
    print(f"\n{'='*60}")
    print(f"Scanning {month_str}")
    print(f"{'='*60}")

    local_path = download_month(y, m)
    collect_sampled_games(local_path, buckets, elo_brackets, target_per_bracket)

    if os.path.exists(local_path):
        os.remove(local_path)
        print(f"Deleted {local_path}")

# Collection summary
print(f"\n{'='*60}")
print("Collection summary:")
for bracket in elo_brackets:
    lo, hi = bracket
    label = f"{lo}-{hi}" if hi < 9999 else f"{lo}+"
    count = len(buckets[bracket])
    print(f"  ELO {label:>9}: {count:,}/{target_per_bracket:,}")
total = sum(len(v) for v in buckets.values())
print(f"  {'Total':>14}: {total:,}")
print(f"{'='*60}")

# Phase 2: Full parse of collected games
all_game_texts = []
for bracket in elo_brackets:
    all_game_texts.extend(buckets[bracket])
del buckets

print(f"\nParsing {len(all_game_texts):,} games...")
games_metadata, blunder_records, move_records = process_collected_games(
    all_game_texts,
    sync_every=SYNC_EVERY,
    vm_dir=VM_DATA_DIR,
    drive_dir=DRIVE_DATA_DIR,
    prefix=OUTPUT_PREFIX,
)

print(f"\nResults:")
print(f"  Games: {len(games_metadata):,}")
print(f"  Blunders: {len(blunder_records):,}")
print(f"  Moves (with eval+clock): {len(move_records):,}")

del all_game_texts

## Save to Google Drive

In [ ]:
# Create DataFrames
df_games = pd.DataFrame(games_metadata)
df_blunders = pd.DataFrame(blunder_records)
df_moves = pd.DataFrame(move_records)

# Preview
print("=== Games Metadata ===")
print(f"Shape: {df_games.shape}")
print(df_games.head())

print("\n=== Blunder Records ===")
print(f"Shape: {df_blunders.shape}")
if not df_blunders.empty:
    print(df_blunders.head())
else:
    print("No blunder records (games may lack eval+clock data)")

print("\n=== Move Records (eval+clock) ===")
print(f"Shape: {df_moves.shape}")
if not df_moves.empty:
    print(df_moves.head())
else:
    print("No move records (games may lack eval+clock data)")

In [ ]:
# Verify saved files on Drive
print("Files on Drive:")
for f in sorted(os.listdir(DRIVE_DATA_DIR)):
    if f.endswith(".csv"):
        path = os.path.join(DRIVE_DATA_DIR, f)
        size_mb = os.path.getsize(path) / 1024**2
        print(f"  {f} ({size_mb:.1f} MB)")

## Summary Statistics

In [ ]:
print("=== Summary ===")
print(f"Total games: {len(df_games):,}")
print(f"\nGame type breakdown:")
if 'game_type' in df_games.columns:
    print(df_games['game_type'].value_counts().to_string())

print(f"\nWinner breakdown:")
print(df_games['winner'].value_counts().to_string())

print(f"\nELO stats (white):")
if df_games['white_rating'].notna().any():
    print(df_games['white_rating'].describe().to_string())

print(f"\nGames with eval data: {df_games['has_eval'].sum():,} ({df_games['has_eval'].mean()*100:.1f}%)")
print(f"Games with clock data: {df_games['has_clock'].sum():,} ({df_games['has_clock'].mean()*100:.1f}%)")

if not df_blunders.empty:
    print(f"\nTotal blunders detected: {len(df_blunders):,}")
    print(f"Blunder rate: {len(df_blunders) / max(len(df_moves), 1) * 100:.2f}% of moves")